# One Forecast, Two Targets

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/brightbandtech/extremeweatherbench/blob/main/notebooks/one_forecast_two_targets.ipynb)

This notebook demonstrates how to evaluate a single forecast against two targets (ERA5 and GHCNh) simultaneously. Optimized for Google Colab with a small demo case.

In [ ]:
!pip install -q extremeweatherbench

In [ ]:
import datetime
import extremeweatherbench as ewb
from extremeweatherbench.cases import IndividualCase
from extremeweatherbench.regions import BoundingBoxRegion

# Mini-case: 2022 India Heat Wave — Colab-optimized
demo_case = IndividualCase(
    case_id_number=9009,
    title="2022 India Heat Wave (demo)",
    start_date=datetime.datetime(2022, 4, 28),
    end_date=datetime.datetime(2022, 5, 1),
    location=BoundingBoxRegion.create_region(
        latitude_min=24.0,
        latitude_max=30.0,
        longitude_min=76.0,
        longitude_max=82.0,
    ),
    event_type="heat_wave",
)
cases = [demo_case]

In [ ]:
forecast = ewb.ZarrForecast(
    source="gs://weatherbench2/datasets/hres/2016-2022-0012-1440x721.zarr",
    name="HRES",
    variable_mapping=ewb.HRES_metadata_variable_mapping,
    storage_options={"remote_options": {"anon": True}},
)

era5_target = ewb.ERA5(
    variables=["surface_air_temperature"]
)
ghcn_target = ewb.GHCN()

In [ ]:
shared_metrics = [
    ewb.metrics.MeanAbsoluteError(
        forecast_variable="surface_air_temperature",
        target_variable="surface_air_temperature",
    ),
    ewb.metrics.MaximumMeanAbsoluteError(
        forecast_variable="surface_air_temperature",
        target_variable="surface_air_temperature",
    ),
]

eval_objects = [
    ewb.EvaluationObject(
        event_type="heat_wave",
        metric_list=shared_metrics,
        target=era5_target,
        forecast=forecast,
    ),
    ewb.EvaluationObject(
        event_type="heat_wave",
        metric_list=shared_metrics,
        target=ghcn_target,
        forecast=forecast,
    ),
]

runner = ewb.evaluation(
    case_metadata=cases,
    evaluation_objects=eval_objects,
)
outputs = runner.run_evaluation()

mae = outputs[outputs["metric"] == "MeanAbsoluteError"]
for source in ["ERA5", "GHCN"]:
    mean_mae = mae[
        mae["target_source"] == source
    ]["value"].mean()
    print(f"{source:6s} mean MAE: {mean_mae:.4f} K")